# Digital Twins - Air Quality Simulation
**Team Members:** Frendl Krisztián & Kozicz Gergő

**Task 1: System Selection**
We have selected **Indoor Air Quality (PM2.5)** as our semester project topic.
This falls under the Monitoring, Reporting, and Verification (MRV) Digital Twin prototype category.

**Task 2: Real Observation Data**
We are using a publicly available dataset from Hugging Face containing Air Quality data:
[Karan-0123/air-quality](https://huggingface.co/datasets/karan-0123/air-quality)

**Task 3: Data Exploration and Simulation**
In this notebook, we will:
1. Load a sample of the historical dataset, including multiple environmental factors (simulated fetching for now).
2. Perform data quality checks.
3. Develop an advanced simulation model based on the Mass Balance Equation for indoor PM2.5, incorporating wind speed and humidity effects.
4. Run 6 What-If Scenarios including the Baseline.
5. Plot the scenarios individually and in a comprehensive comparison plot.

## 1. Data Loading and Initial Exploration
We generate a synthetic dataset mimicking 24 hours of outdoor PM2.5 data, along with wind speed (m/s) and relative humidity (%).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import huggingface_hub as hf_hub


df = pd.read_csv("hf://datasets/karan-0123/air-quality/air_quality.csv")
df = df[:24]

display(df.head())

## 2. Advanced Simulation Logic
We use an extended Mass Balance Equation:
$$PM_{in}(t) = PM_{in}(t-1) + \Delta t \cdot [ k(v) \cdot (PM_{out}(t) - PM_{in}(t-1)) - \gamma(h) \cdot PM_{in}(t-1) + \alpha \cdot Source - \beta \cdot Purifier ]$$

**Parameters & Functions:**
*   $PM_{in}$: Indoor air pollution.
*   $PM_{out}$: Outdoor air pollution.
*   $k(v) = k_{base} + c_w \cdot v$: Infiltration rate, where $k_{base}$ is base infiltration, $c_w$ is the wind coefficient, and $v$ is wind speed.
*   $\gamma(h) = c_h \cdot h$: Deposition rate (particles settling), which increases with high humidity $h$ (due to particle growth/heaviness).
*   $\alpha \cdot Source$: Internal pollution source (e.g., cooking).
*   $\beta \cdot Purifier$: Air purifier efficiency (CADR).

In [ ]:
def simulate_advanced_indoor_aq(data, k_base=0.1, c_w=0.05, c_h=0.001, alpha_source=0.0, beta_purifier=0.0):
    """
    Simulates indoor PM2.5 based on an advanced Mass Balance Equation.
    """
    delta_t = 1
    PM_in_initial = 15.0
    PM_in_list = [PM_in_initial]
    
    for i in range(len(data)):
        PM_in_prev = PM_in_list[-1]
        PM_out = data['pm25'].iloc[i]
        v = data['wind'].iloc[i]
        h = data['rh'].iloc[i]
        
        # Dynamic infiltration based on wind speed
        k_v = k_base + c_w * v
        
        # Dynamic deposition based on humidity
        gamma_h = c_h * h
        
        # Calculate changes
        change = delta_t * (k_v * (PM_out - PM_in_prev) - gamma_h * PM_in_prev + alpha_source - beta_purifier)
        
        # Update state
        PM_in_new = max(0, PM_in_prev + change)
        PM_in_list.append(PM_in_new)
        
    return PM_in_list[1:]

## 3. What-If Scenarios

1.  **Baseline:** Normal conditions ($k_{base}=0.1, c_w=0.05, c_h=0.001$).
2.  **Scenario A (Drafty House):** High base infiltration and wind sensitivity ($k_{base}=0.3, c_w=0.1$).
3.  **Scenario B (Humidifier ON):** Simulate high indoor humidity artificially increasing deposition (we'll boost $c_h$ to 0.005 for effect).
4.  **Scenario C (Cooking Event):** Strong internal source added.
5.  **Scenario D (Cooking + Purifier):** Internal source present, Air Purifier is running.
6.  **Scenario E (Stormy Draft):** Cooking event, purifier ON, but extreme drafty house ($k_{base}=0.5$). Testing if purifier can keep up with high infiltration.

In [ ]:
# Run Scenarios
baseline = simulate_advanced_indoor_aq(df)
scen_A_drafty = simulate_advanced_indoor_aq(df, k_base=0.3, c_w=0.1)
scen_B_humidifier = simulate_advanced_indoor_aq(df, c_h=0.005) # Increased deposition
scen_C_cooking = simulate_advanced_indoor_aq(df, alpha_source=12.0)
scen_D_purifier = simulate_advanced_indoor_aq(df, alpha_source=12.0, beta_purifier=15.0)
scen_E_stormy = simulate_advanced_indoor_aq(df, k_base=0.5, c_w=0.2, alpha_source=12.0, beta_purifier=15.0)

## 4. Plotting Scenarios Individually

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(16, 15))
time_labels = df['datetime'].apply(lambda x: x.split(' ')[1][:5])

# Baseline vs Drafty House
axs[0, 0].plot(time_labels, df['pm25'], label='Outdoor PM2.5', linestyle='--', color='gray')
axs[0, 0].plot(time_labels, baseline, label='Baseline')
axs[0, 0].plot(time_labels, scen_A_drafty, label='Scenario A: Drafty', color='red')
axs[0, 0].set_title('Baseline vs Drafty House (Wind Effect)')
axs[0, 0].set_ylabel('PM2.5 (ug/m3)')
axs[0, 0].legend()
axs[0, 0].tick_params(axis='x', rotation=45)

# Baseline vs Humidifier
axs[0, 1].plot(time_labels, df['pm25'], label='Outdoor PM2.5', linestyle='--', color='gray')
axs[0, 1].plot(time_labels, baseline, label='Baseline')
axs[0, 1].plot(time_labels, scen_B_humidifier, label='Scenario B: Humidifier (High Deposition)', color='cyan')
axs[0, 1].set_title('Baseline vs Increased Humidity')
axs[0, 1].legend()
axs[0, 1].tick_params(axis='x', rotation=45)

# Baseline vs Cooking Event
axs[1, 0].plot(time_labels, df['pm25'], label='Outdoor PM2.5', linestyle='--', color='gray')
axs[1, 0].plot(time_labels, baseline, label='Baseline')
axs[1, 0].plot(time_labels, scen_C_cooking, label='Scenario C: Cooking (+Source)', color='orange')
axs[1, 0].set_title('Baseline vs Cooking Event')
axs[1, 0].set_ylabel('PM2.5 (ug/m3)')
axs[1, 0].legend()
axs[1, 0].tick_params(axis='x', rotation=45)

# Cooking vs Purifier
axs[1, 1].plot(time_labels, df['pm25'], label='Outdoor PM2.5', linestyle='--', color='gray')
axs[1, 1].plot(time_labels, scen_C_cooking, label='Cooking (No Purifier)', color='orange')
axs[1, 1].plot(time_labels, scen_D_purifier, label='Scenario D: Cooking + Purifier ON', color='green')
axs[1, 1].set_title('Effect of Air Purifier during Cooking')
axs[1, 1].legend()
axs[1, 1].tick_params(axis='x', rotation=45)

# Purifier Normal vs Stormy Draft
axs[2, 0].plot(time_labels, df['pm25'], label='Outdoor PM2.5', linestyle='--', color='gray')
axs[2, 0].plot(time_labels, scen_D_purifier, label='Purifier ON (Normal House)', color='green')
axs[2, 0].plot(time_labels, scen_E_stormy, label='Scenario E: Purifier ON + Stormy Draft', color='purple')
axs[2, 0].set_title('Purifier Performance in Drafty Conditions')
axs[2, 0].set_ylabel('PM2.5 (ug/m3)')
axs[2, 0].legend()
axs[2, 0].tick_params(axis='x', rotation=45)

axs[2, 1].axis('off') # Hide empty subplot

plt.tight_layout()
plt.show()

## 5. Comparing the results

In [ ]:
plt.figure(figsize=(16, 8))
plt.plot(time_labels, df['pm25'], label='Outdoor PM2.5', linestyle='--', color='black', linewidth=2)
plt.plot(time_labels, baseline, label='Baseline', linewidth=2)
plt.plot(time_labels, scen_A_drafty, label='A: Drafty House (Wind Effect)')
plt.plot(time_labels, scen_B_humidifier, label='B: Humidifier (High Deposition)')
plt.plot(time_labels, scen_C_cooking, label='C: Internal Source (Cooking)')
plt.plot(time_labels, scen_D_purifier, label='D: Internal Source + Purifier ON')
plt.plot(time_labels, scen_E_stormy, label='E: Purifier ON + Stormy Draft', linewidth=2)

plt.title('Advanced Indoor Air Quality (PM2.5) Simulation - Scenario Comparison', fontsize=16)
plt.xlabel('Time of Day', fontsize=12)
plt.ylabel('PM2.5 Concentration (ug/m3)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()